# Run the experiments: DNA language models vs. CADD

**How to use this notebook**

1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Menu **Runtime → Run all**. Allow access to Google Drive when asked.
3. Keep this browser tab open. The whole run takes about **2–3 hours**.
4. If Colab disconnects, just reconnect and **Run all** again: finished steps are saved in your Google Drive (`MyDrive/glm-variant-pathogenicity`) and are skipped.
5. At the end, download **results_bundle.zip** (last cell) and send it to Claude to put the numbers into the slides.

This is the *reduced* version of the Module C plan (`configs/colab.yaml`): a random subset of each gene-level split (24,000 train / 3,000 validation / 4,000 test variants), one seed, a fixed learning rate and up to 3 epochs.

## 1. Check the GPU

In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again."

## 2. Connect Google Drive (results are saved there)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/glm-variant-pathogenicity

## 3. Install libraries
DNABERT-2's optional Triton kernel does not work with current Triton, so it is removed and DNABERT-2 falls back to standard PyTorch attention.

In [ ]:
!pip uninstall -y -q triton
!pip install -q "transformers>=4.44,<4.48" "peft>=0.12,<0.14" pysam einops tabulate pyarrow

## 4. Get the code from GitHub
The cells after this one update a few files with the versions used for this GPU run (more robust model loading, float16 support for the T4, remote CADD lookup).

In [ ]:
%cd /content
!rm -rf glm-variant-pathogenicity
!git clone -q https://github.com/Yasmin-maker1/glm-variant-pathogenicity.git
%cd /content/glm-variant-pathogenicity
!mkdir -p notebooks

In [ ]:
%%writefile /content/glm-variant-pathogenicity/src/glmvar/models/loading.py
"""Load a pretrained backbone + tokenizer and wrap it for one of two training modes.

Modes (Section III.D):
  * ``finetune`` - LoRA for transformer models, full fine-tuning for HyenaDNA
  * ``frozen``   - backbone weights fixed, only the classification head trains

Each of the three checkpoints ships its own ``trust_remote_code`` model, and
they do not all register the same Auto* classes, so loading tries AutoModel
first and falls back to the masked-LM / causal-LM class and its base model.
"""

from __future__ import annotations

import logging

from .classifier import PairClassifier, hidden_size_of
from .registry import ModelSpec

LOG = logging.getLogger(__name__)


def _extra_kwargs(spec: ModelSpec) -> dict:
    kw = {"trust_remote_code": True}
    if spec.key.startswith("dnabert2"):
        # Known DNABERT-2 workaround for newer transformers versions:
        # pass the standard BertConfig explicitly.
        from transformers.models.bert.configuration_bert import BertConfig

        kw["config"] = BertConfig.from_pretrained(spec.hf_id)
    return kw


def disable_dnabert2_triton() -> int:
    """Make DNABERT-2 use its plain PyTorch attention instead of its Triton kernel.

    DNABERT-2's optional flash-attention kernel was written for an old Triton
    API (``tl.dot(..., trans_b=True)``) and fails to compile with current
    Triton. Its code falls back to standard attention when the module-level
    ``flash_attn_qkvpacked_func`` is None, so we set that after loading.
    Returns the number of modules patched.
    """
    import sys

    patched = 0
    for name, mod in list(sys.modules.items()):
        if name.endswith("bert_layers") and getattr(mod, "flash_attn_qkvpacked_func", None) is not None:
            mod.flash_attn_qkvpacked_func = None
            patched += 1
    return patched


def load_tokenizer(spec: ModelSpec):
    from transformers import AutoTokenizer

    return AutoTokenizer.from_pretrained(spec.hf_id, trust_remote_code=True)


def load_backbone(spec: ModelSpec):
    from transformers import AutoModel, AutoModelForCausalLM, AutoModelForMaskedLM

    kw = _extra_kwargs(spec)
    errors = []
    try:
        model = AutoModel.from_pretrained(spec.hf_id, **kw)
        if disable_dnabert2_triton():
            LOG.info("DNABERT-2: using PyTorch attention (Triton kernel disabled)")
        return model
    except Exception as exc:  # noqa: BLE001 - remote code raises many types
        errors.append(f"AutoModel: {exc}")
    for cls in (AutoModelForMaskedLM, AutoModelForCausalLM):
        try:
            model = cls.from_pretrained(spec.hf_id, **kw)
            disable_dnabert2_triton()
            LOG.info("loaded %s via %s; using its base model", spec.hf_id, cls.__name__)
            return model.base_model
        except Exception as exc:  # noqa: BLE001
            errors.append(f"{cls.__name__}: {exc}")
    raise RuntimeError(f"could not load {spec.hf_id}:\n" + "\n".join(errors))


def apply_lora(backbone, spec: ModelSpec, r: int = 8, alpha: int = 16, dropout: float = 0.05):
    from peft import LoraConfig, get_peft_model

    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, target_modules=list(spec.lora_targets), bias="none")
    return get_peft_model(backbone, cfg)


def build_classifier(spec: ModelSpec, mode: str, lora_cfg: dict | None = None, head_cfg: dict | None = None, backbone=None):
    """Return a PairClassifier ready to train. ``backbone`` can be injected for tests."""
    lora_cfg = lora_cfg or {}
    head_cfg = head_cfg or {}
    backbone = backbone if backbone is not None else load_backbone(spec)
    hidden = hidden_size_of(backbone)
    if mode == "frozen":
        return PairClassifier(backbone, hidden, freeze_backbone=True, **head_cfg)
    if mode != "finetune":
        raise ValueError(f"unknown mode {mode!r}")
    if spec.finetune == "lora":
        backbone = apply_lora(backbone, spec, **lora_cfg)
    return PairClassifier(backbone, hidden, freeze_backbone=False, **head_cfg)


def count_trainable(model) -> tuple[int, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total

In [ ]:
%%writefile /content/glm-variant-pathogenicity/src/glmvar/models/classifier.py
"""Reference/alternate pair classifier shared by all three models (Section III.D).

Both sequences go through the same backbone. Each is reduced to one vector by
averaging token embeddings near the variant; [ref, alt, alt - ref] is fed to a
small feed-forward head that outputs a pathogenicity logit.
"""

from __future__ import annotations

import inspect

import torch
from torch import nn


def backbone_hidden_states(backbone: nn.Module, input_ids: torch.Tensor, attention_mask: torch.Tensor | None) -> torch.Tensor:
    """Return [batch, tokens, hidden] for HF models with differing output types."""
    kwargs = {"input_ids": input_ids}
    # A PEFT wrapper exposes forward(*args, **kwargs); read the real signature
    # from the wrapped model so attention_mask is still passed through.
    target = backbone.get_base_model() if hasattr(backbone, "get_base_model") else backbone
    params = inspect.signature(target.forward).parameters
    if attention_mask is not None and "attention_mask" in params:
        kwargs["attention_mask"] = attention_mask
    if "output_hidden_states" in params:
        kwargs["output_hidden_states"] = True
    out = backbone(**kwargs)
    if hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
        return out.last_hidden_state
    if hasattr(out, "hidden_states") and out.hidden_states is not None:
        return out.hidden_states[-1]
    if isinstance(out, (tuple, list)):
        return out[0]
    return out


def masked_mean(h: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    m = mask.unsqueeze(-1).to(h.dtype)
    return (h * m).sum(1) / m.sum(1).clamp(min=1.0)


class PairClassifier(nn.Module):
    def __init__(self, backbone: nn.Module, hidden_size: int, head_hidden: int = 256, dropout: float = 0.1, freeze_backbone: bool = False):
        super().__init__()
        self.backbone = backbone
        self.freeze_backbone = freeze_backbone
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
        self.head = nn.Sequential(
            nn.Linear(3 * hidden_size, head_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, 1),
        )

    def embed(self, input_ids, attention_mask, pool_mask):
        if self.freeze_backbone:
            with torch.no_grad():
                h = backbone_hidden_states(self.backbone, input_ids, attention_mask)
        else:
            h = backbone_hidden_states(self.backbone, input_ids, attention_mask)
        return masked_mean(h.float(), pool_mask)

    def forward(self, ref_ids, ref_attn, ref_pool, alt_ids, alt_attn, alt_pool) -> torch.Tensor:
        # One backbone pass over the stacked ref+alt batch.
        b = ref_ids.shape[0]
        emb = self.embed(
            torch.cat([ref_ids, alt_ids]),
            torch.cat([ref_attn, alt_attn]),
            torch.cat([ref_pool, alt_pool]),
        )
        r, a = emb[:b], emb[b:]
        return self.head(torch.cat([r, a, a - r], dim=-1)).squeeze(-1)


def hidden_size_of(backbone: nn.Module) -> int:
    cfg = getattr(backbone, "config", None)
    for attr in ("hidden_size", "d_model", "n_embd"):
        if cfg is not None and getattr(cfg, attr, None):
            return int(getattr(cfg, attr))
    raise ValueError("could not infer hidden size; pass it explicitly")

In [ ]:
%%writefile /content/glm-variant-pathogenicity/src/glmvar/train.py
"""Fine-tune (or train a head on a frozen) genomic language model on variant pairs.

Implements the training settings of Module C, Section III.D:
AdamW, mixed precision, class-weighted BCE, batch 32 (with gradient accumulation),
up to 10 epochs, early stopping after 2 epochs without validation-AUROC gain.
"""

from __future__ import annotations

import copy
import logging
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

from .models.tokens import pool_mask
from .utils import ResourceTracker, save_json, set_seed

LOG = logging.getLogger(__name__)


class VariantPairDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.ref = df["ref_seq"].tolist()
        self.alt = df["alt_seq"].tolist()
        self.y = df["label"].astype(np.float32).to_numpy()
        self.ids = df["variant_id"].tolist()

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, i):
        return self.ref[i], self.alt[i], self.y[i]


class PairCollator:
    """Tokenizes ref/alt windows and builds the pooling mask around the variant."""

    def __init__(self, tokenizer, pool_radius: int = 32):
        self.tok = tokenizer
        self.radius = pool_radius

    def _encode(self, seqs: list[str]):
        enc = self.tok(seqs, padding="longest", return_tensors="pt", add_special_tokens=True)
        pools = []
        for ids, seq in zip(enc["input_ids"].tolist(), seqs):
            tokens = self.tok.convert_ids_to_tokens(ids)
            pools.append(pool_mask(tokens, center=len(seq) // 2, radius=self.radius))
        attn = enc.get("attention_mask", torch.ones_like(enc["input_ids"]))
        return enc["input_ids"], attn, torch.tensor(pools, dtype=torch.float32)

    def __call__(self, batch):
        ref, alt, y = zip(*batch)
        # Encode ref and alt together so both are padded to the same length
        # (BPE can split the two windows into a different number of tokens).
        ids, attn, pool = self._encode(list(ref) + list(alt))
        n = len(ref)
        r_ids, r_attn, r_pool = ids[:n], attn[:n], pool[:n]
        a_ids, a_attn, a_pool = ids[n:], attn[n:], pool[n:]
        return {
            "ref_ids": r_ids, "ref_attn": r_attn, "ref_pool": r_pool,
            "alt_ids": a_ids, "alt_attn": a_attn, "alt_pool": a_pool,
            "labels": torch.tensor(y, dtype=torch.float32),
        }


def _to(batch: dict, device) -> dict:
    return {k: v.to(device) for k, v in batch.items()}


_AMP = {"enabled": True}


def _amp_dtype(device):
    """bfloat16 on Ampere or newer, float16 on older GPUs (e.g. a T4), off on CPU."""
    if device.type != "cuda" or not _AMP["enabled"]:
        return None
    major, _ = torch.cuda.get_device_capability(device)
    return torch.bfloat16 if major >= 8 else torch.float16


def _autocast(device):
    dtype = _amp_dtype(device)
    if dtype is None:
        return torch.autocast("cpu", enabled=False)
    return torch.autocast("cuda", dtype=dtype)


@torch.no_grad()
def predict(model, loader, device) -> np.ndarray:
    model.eval()
    out = []
    for batch in loader:
        batch = _to(batch, device)
        batch.pop("labels")
        with _autocast(device):
            logits = model(**batch)
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out) if out else np.array([])


def train_model(model, tokenizer, df: pd.DataFrame, cfg: dict, out_dir: str | Path, seed: int = 0, device: str | None = None) -> dict:
    """Train with early stopping on validation AUROC; write val/test predictions.

    ``df`` must contain columns: variant_id, split, label, ref_seq, alt_seq.
    Returns a summary dict (also saved as ``summary.json``).
    """
    set_seed(seed)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    model.to(device)

    t = cfg.get("training", {})
    _AMP["enabled"] = bool(t.get("amp", True))
    micro_bs = int(t.get("micro_batch_size", 8))
    eff_bs = int(t.get("batch_size", 32))
    accum = max(1, eff_bs // micro_bs)
    epochs = int(t.get("epochs", 10))
    patience = int(t.get("patience", 2))
    lr = float(t.get("learning_rate", 3e-5))
    head_lr = float(t.get("head_learning_rate", 1e-3))
    wd = float(t.get("weight_decay", 0.01))
    collate = PairCollator(tokenizer, pool_radius=int(t.get("pool_radius", 32)))

    parts = {s: df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test")}
    train_loader = DataLoader(VariantPairDataset(parts["train"]), batch_size=micro_bs, shuffle=True, collate_fn=collate, num_workers=int(t.get("num_workers", 0)))
    eval_loaders = {s: DataLoader(VariantPairDataset(parts[s]), batch_size=micro_bs * 2, shuffle=False, collate_fn=collate) for s in ("val", "test")}

    # Class weighting for the pathogenic/benign imbalance.
    y = parts["train"]["label"].to_numpy()
    pos_weight = torch.tensor([(y == 0).sum() / max(1, (y == 1).sum())], device=device, dtype=torch.float32)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    head_params = list(model.head.parameters())
    head_ids = {id(p) for p in head_params}
    bb_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": head_params, "lr": head_lr}]
    if bb_params:
        groups.append({"params": bb_params, "lr": lr})
    opt = torch.optim.AdamW(groups, weight_decay=wd)

    # float16 (e.g. a Colab T4) needs loss scaling; bfloat16 and CPU do not.
    scaler = torch.amp.GradScaler("cuda", enabled=_amp_dtype(device) == torch.float16)
    all_params = [p for g in groups for p in g["params"]]

    tracker = ResourceTracker()
    best_auc, best_state, bad_epochs, history = -1.0, None, 0, []
    with tracker.track():
        for epoch in range(1, epochs + 1):
            model.train()
            opt.zero_grad(set_to_none=True)
            running = 0.0
            for step, batch in enumerate(train_loader, 1):
                batch = _to(batch, device)
                labels = batch.pop("labels")
                with _autocast(device):
                    logits = model(**batch)
                loss = loss_fn(logits.float(), labels) / accum
                scaler.scale(loss).backward()
                running += loss.item() * accum
                if step % accum == 0 or step == len(train_loader):
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(all_params, 1.0)
                    scaler.step(opt)
                    scaler.update()
                    opt.zero_grad(set_to_none=True)
                if step % 200 == 0:
                    LOG.info("epoch %d  step %d/%d  loss %.4f", epoch, step, len(train_loader), running / step)
            val_pred = predict(model, eval_loaders["val"], device)
            val_auc = roc_auc_score(parts["val"]["label"], val_pred) if parts["val"]["label"].nunique() > 1 else float("nan")
            history.append({"epoch": epoch, "train_loss": running / max(1, len(train_loader)), "val_auroc": val_auc})
            LOG.info("epoch %d  loss %.4f  val AUROC %.4f", epoch, history[-1]["train_loss"], val_auc)
            if val_auc > best_auc:
                best_auc, bad_epochs = val_auc, 0
                best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    LOG.info("early stopping after epoch %d", epoch)
                    break

    if best_state is not None:
        model.load_state_dict(best_state)

    preds = {}
    for split in ("val", "test"):
        start = time.perf_counter()
        p = predict(model, eval_loaders[split], device)
        elapsed = time.perf_counter() - start
        preds[split] = p
        frame = parts[split][["variant_id", "label"]].copy()
        frame["score"] = p
        frame["split"] = split
        frame.to_csv(out_dir / f"predictions_{split}.csv", index=False)
        if split == "test":
            variants_per_second = len(p) / elapsed if elapsed > 0 else float("nan")

    summary = {
        "seed": seed,
        "best_val_auroc": best_auc,
        "epochs_run": len(history),
        "history": history,
        "variants_per_second": variants_per_second,
        "n_train": len(parts["train"]),
        **tracker.as_dict(),
    }
    save_json(summary, out_dir / "summary.json")
    return summary

In [ ]:
%%writefile /content/glm-variant-pathogenicity/src/glmvar/cadd.py
"""CADD v1.7 baseline (Section III.E): look up precomputed PHRED scores.

Three options:
  1. ``lookup_tabix`` on the full ``whole_genome_SNVs.tsv.gz`` (+ .tbi). The path
     may be a local file (~80 GB download) **or the https URL** of the file on
     the CADD server: htslib then fetches only the small blocks it needs.
  2. ``lookup_table`` on a smaller TSV you scored with the CADD web service
     (upload ``data/processed/variants.vcf`` and download the result).
CADD is not retrained; only its decision threshold is chosen on validation data.
"""

from __future__ import annotations

import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

CADD_URL = "https://kircherlab.bihealth.org/download/CADD/v1.7/GRCh38/whole_genome_SNVs.tsv.gz"


def lookup_tabix(df: pd.DataFrame, tsv_gz: str | Path, column: str = "PHRED", threads: int = 8) -> np.ndarray:
    import pysam

    local = threading.local()

    def handle():
        if not hasattr(local, "tbx"):
            local.tbx = pysam.TabixFile(str(tsv_gz))
        return local.tbx

    tbx0 = handle()
    header = None
    for line in tbx0.header:
        if line.startswith("#Chrom") or line.startswith("#CHROM"):
            header = line.lstrip("#").split("\t")
    col = header.index(column) if header else 5

    rows = list(df[["chrom", "pos", "ref", "alt"]].itertuples(index=False))

    def one(r):
        for attempt in range(3):
            try:
                for row in handle().fetch(str(r.chrom), r.pos - 1, r.pos):
                    f = row.split("\t")
                    if int(f[1]) == r.pos and f[2] == r.ref and f[3] == r.alt:
                        return float(f[col])
                return np.nan
            except (OSError, ValueError):
                if attempt == 2:
                    return np.nan
                local.__dict__.pop("tbx", None)  # reopen after a network hiccup
        return np.nan

    with ThreadPoolExecutor(max_workers=threads) as pool:
        return np.array(list(pool.map(one, rows)), dtype=float)


def lookup_table(df: pd.DataFrame, table: str | Path, column: str = "PHRED") -> np.ndarray:
    cadd = pd.read_csv(table, sep="\t", skiprows=lambda i: i == 0 and _is_version_line(table), dtype={"#Chrom": str, "Chrom": str})
    cadd = cadd.rename(columns={"#Chrom": "Chrom"})
    cadd["variant_id"] = cadd["Chrom"].astype(str).str.removeprefix("chr") + "-" + cadd["Pos"].astype(str) + "-" + cadd["Ref"] + "-" + cadd["Alt"]
    m = df[["variant_id"]].merge(cadd[["variant_id", column]].drop_duplicates("variant_id"), on="variant_id", how="left")
    return m[column].to_numpy(dtype=float)


def _is_version_line(path) -> bool:
    import gzip

    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt") as fh:
        return fh.readline().startswith("## CADD")

In [ ]:
%%writefile /content/glm-variant-pathogenicity/src/glmvar/zeroshot.py
"""Zero-shot masked-language-model scores (Section III.D, second reference point).

For DNABERT-2 and the Nucleotide Transformer we mask the token that contains
the variant base and compare the model's log-probabilities of the reference
token and of the same token with the alternate base:

    LLR = log p(alt_token) - log p(ref_token)

A strongly negative LLR means the model finds the variant unlikely, which we
treat as evidence of pathogenicity, so the score reported is ``-LLR``.
If the alternate token string is not in the vocabulary (possible with BPE),
the score is NaN and the coverage is reported.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
import torch

from .models.tokens import token_index_at, token_spans


def load_mlm(hf_id: str):
    from transformers import AutoModelForMaskedLM, AutoTokenizer

    kw = {"trust_remote_code": True}
    if "DNABERT-2" in hf_id:
        from transformers.models.bert.configuration_bert import BertConfig

        kw["config"] = BertConfig.from_pretrained(hf_id)
    tok = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=True)
    model = AutoModelForMaskedLM.from_pretrained(hf_id, **kw)
    from .models.loading import disable_dnabert2_triton

    disable_dnabert2_triton()
    return model, tok


@torch.no_grad()
def llr_scores(model, tokenizer, df: pd.DataFrame, batch_size: int = 16, device: str | None = None) -> np.ndarray:
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    model.to(device).eval()
    unk = tokenizer.unk_token_id
    scores = np.full(len(df), np.nan, dtype=np.float64)
    rows = list(df.itertuples(index=False))
    for start in range(0, len(rows), batch_size):
        chunk = rows[start : start + batch_size]
        enc = tokenizer([r.ref_seq for r in chunk], return_tensors="pt", padding="longest")
        ids = enc["input_ids"].clone()
        targets = []  # (row_in_batch, token_idx, ref_id, alt_id)
        for b, r in enumerate(chunk):
            tokens = tokenizer.convert_ids_to_tokens(ids[b].tolist())
            center = len(r.ref_seq) // 2
            ti = token_index_at(tokens, center)
            if ti is None:
                continue
            s, _ = token_spans(tokens)[ti]
            ref_tok = tokens[ti]
            offset = center - s
            alt_tok = ref_tok[:offset] + r.alt + ref_tok[offset + 1 :]
            alt_id = tokenizer.convert_tokens_to_ids(alt_tok)
            if alt_id is None or alt_id == unk:
                continue
            targets.append((b, ti, int(ids[b, ti]), int(alt_id)))
            ids[b, ti] = tokenizer.mask_token_id
        if not targets:
            continue
        kwargs = {"input_ids": ids.to(device)}
        if "attention_mask" in enc:
            kwargs["attention_mask"] = enc["attention_mask"].to(device)
        out = model(**kwargs)
        logits = out.logits if hasattr(out, "logits") else out[0]
        if logits.dim() != 3 or logits.shape[1] != ids.shape[1]:
            raise RuntimeError(f"unexpected MLM logits shape {tuple(logits.shape)}; zero-shot not supported for this model")
        logp = torch.log_softmax(logits.float(), dim=-1)
        for b, ti, ref_id, alt_id in targets:
            llr = (logp[b, ti, alt_id] - logp[b, ti, ref_id]).item()
            scores[start + b] = -llr
    return scores

In [ ]:
%%writefile /content/glm-variant-pathogenicity/src/glmvar/cli.py
"""Command-line entry point: ``glmvar <command> [options]``.

  glmvar build-dataset  --config configs/default.yaml
  glmvar train          --config configs/default.yaml --model dnabert2 --mode finetune --seed 0
  glmvar zeroshot       --config configs/default.yaml --model nt_v2_250m
  glmvar cadd           --config configs/default.yaml
  glmvar report         --config configs/default.yaml
"""

from __future__ import annotations

import argparse
import logging
from pathlib import Path

import pandas as pd

from .utils import load_config, save_json, setup_logging

LOG = logging.getLogger("glmvar")


def _variants_path(cfg, window: int | None = None) -> Path:
    w = window or cfg["data"]["window_bp"]
    return Path(cfg["paths"]["processed"]) / f"variants_w{w}.parquet"


def cmd_build_dataset(cfg, args):
    from .data.clinvar import load_clinvar
    from .data.sequences import ReferenceGenome, add_sequences
    from .data.split import check_no_leakage, gene_split

    d = cfg["data"]
    LOG.info("parsing ClinVar %s", d["clinvar_vcf"])
    df = load_clinvar(d["clinvar_vcf"], min_stars=d.get("min_review_stars", 1), drop_multi_gene=d.get("drop_multi_gene", True))
    LOG.info("%d labeled SNVs (%d pathogenic, %d benign)", len(df), int(df.label.sum()), int((df.label == 0).sum()))
    if d.get("vep_tsv"):
        from .data.vep import merge_vep, read_vep_tab

        df = merge_vep(df, read_vep_tab(d["vep_tsv"]))
    df = gene_split(df, tuple(d["split_fractions"]), seed=d["split_seed"])
    check_no_leakage(df)
    caps = d.get("max_variants_per_split")
    if caps:
        # Reduced-compute run: random sample within each gene-level split
        # (the split itself is unchanged, so there is still no gene leakage).
        df = pd.concat([g.sample(min(len(g), int(caps[s])), random_state=d["split_seed"]) if s in caps else g
                        for s, g in df.groupby("split")]).reset_index(drop=True)
        LOG.info("subsampled to %s", df.split.value_counts().to_dict())
    genome = ReferenceGenome(d["reference_fasta"])
    windows = sorted({d["window_bp"], *d.get("extra_windows_bp", [])}) if args.all_windows else [d["window_bp"]]
    stats = {}
    for w in windows:
        out = add_sequences(df, genome, w)
        path = _variants_path(cfg, w)
        path.parent.mkdir(parents=True, exist_ok=True)
        out.to_parquet(path, index=False)
        stats[w] = {"n": len(out), "dropped_ref_mismatch": out.attrs.get("n_dropped_ref_mismatch", 0),
                    "by_split": out.groupby("split").label.agg(["size", "sum"]).to_dict()}
        LOG.info("window %d bp -> %s (%d variants)", w, path, len(out))
    save_json(stats, Path(cfg["paths"]["processed"]) / "dataset_stats.json")


def cmd_train(cfg, args):
    from .data.split import subsample_train_genes
    from .models import loading
    from .models.registry import get_spec
    from .train import train_model

    spec = get_spec(args.model)
    name = args.run_name or f"{spec.key}_{args.mode}" + (f"_w{args.window}" if args.window else "") + (f"_frac{args.train_fraction:g}" if args.train_fraction < 1 else "")
    out = Path(cfg["paths"]["runs"]) / name / f"seed{args.seed}"
    if args.skip_existing and (out / "summary.json").exists():
        LOG.info("skipping %s (already done)", out)
        return
    df = pd.read_parquet(_variants_path(cfg, args.window))
    if args.train_fraction < 1:
        df = subsample_train_genes(df, args.train_fraction, seed=args.seed)
    if args.limit:
        df = df.groupby("split", group_keys=False).apply(lambda g: g.sample(min(len(g), args.limit), random_state=0))
    if args.learning_rate:
        cfg["training"]["learning_rate"] = args.learning_rate
    if spec.key.startswith("hyenadna"):
        # HyenaDNA's FFT long convolutions are not reliable in float16; the
        # model is small, so it trains in full precision.
        cfg["training"]["amp"] = False
    tok = loading.load_tokenizer(spec)
    model = loading.build_classifier(spec, args.mode, lora_cfg=cfg.get("lora"), head_cfg=cfg.get("head"))
    trainable, total = loading.count_trainable(model)
    LOG.info("%s/%s: %.2fM trainable of %.1fM parameters", spec.key, args.mode, trainable / 1e6, total / 1e6)
    summary = train_model(model, tok, df, cfg, out, seed=args.seed)
    summary.update({"model": spec.key, "hf_id": spec.hf_id, "mode": args.mode, "trainable_params": trainable,
                    "total_params": total, "train_fraction": args.train_fraction, "window_bp": args.window or cfg["data"]["window_bp"]})
    save_json(summary, out / "summary.json")


def cmd_zeroshot(cfg, args):
    from .models.registry import get_spec
    from .zeroshot import llr_scores, load_mlm

    spec = get_spec(args.model)
    if not spec.supports_mlm_zeroshot:
        raise SystemExit(f"{spec.key} is not a masked language model; zero-shot LLR is not defined")
    df = pd.read_parquet(_variants_path(cfg))
    df = df[df.split.isin(["val", "test"])].reset_index(drop=True)
    model, tok = load_mlm(spec.hf_id)
    df["score"] = llr_scores(model, tok, df, batch_size=args.batch_size)
    LOG.info("zero-shot coverage: %.1f%%", 100 * df.score.notna().mean())
    out = Path(cfg["paths"]["runs"]) / f"{spec.key}_zeroshot" / "seed0"
    out.mkdir(parents=True, exist_ok=True)
    for s in ("val", "test"):
        df[df.split == s][["variant_id", "label", "score", "split"]].to_csv(out / f"predictions_{s}.csv", index=False)


def cmd_cadd(cfg, args):
    from .cadd import lookup_table, lookup_tabix

    c = cfg["cadd"]
    df = pd.read_parquet(_variants_path(cfg))
    df = df[df.split.isin(["val", "test"])].reset_index(drop=True)
    if c.get("scored_table"):
        df["score"] = lookup_table(df, c["scored_table"], c.get("column", "PHRED"))
    else:
        df["score"] = lookup_tabix(df, c["whole_genome_tsv"], c.get("column", "PHRED"))
    LOG.info("CADD coverage: %.1f%%", 100 * df.score.notna().mean())
    out = Path(cfg["paths"]["runs"]) / "cadd" / "seed0"
    out.mkdir(parents=True, exist_ok=True)
    for s in ("val", "test"):
        df[df.split == s][["variant_id", "label", "score", "split"]].to_csv(out / f"predictions_{s}.csv", index=False)


def cmd_export_vcf(cfg, args):
    from .data.vep import write_vcf

    df = pd.read_parquet(_variants_path(cfg))
    LOG.info("wrote %s", write_vcf(df, Path(cfg["paths"]["processed"]) / "variants.vcf"))


def cmd_report(cfg, args):
    from .evaluate import write_report
    from .plots import plot_all

    tables = write_report(cfg["paths"]["runs"], _variants_path(cfg), cfg["paths"]["reports"], n_boot=cfg["evaluation"]["bootstrap"])
    plot_all(cfg["paths"]["runs"], cfg["paths"]["reports"])
    print(tables["summary"].to_string(index=False))


def main(argv=None):
    p = argparse.ArgumentParser(prog="glmvar", description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--config", default="configs/default.yaml")
    p.add_argument("--log-level", default="INFO")
    sub = p.add_subparsers(dest="command", required=True)

    b = sub.add_parser("build-dataset")
    b.add_argument("--all-windows", action="store_true", help="also build the long windows for HyenaDNA")
    t = sub.add_parser("train")
    t.add_argument("--model", required=True)
    t.add_argument("--mode", choices=["finetune", "frozen"], default="finetune")
    t.add_argument("--seed", type=int, default=0)
    t.add_argument("--learning-rate", type=float)
    t.add_argument("--train-fraction", type=float, default=1.0)
    t.add_argument("--window", type=int, help="window length in bp (default: data.window_bp)")
    t.add_argument("--limit", type=int, help="debug: cap variants per split")
    t.add_argument("--run-name")
    t.add_argument("--skip-existing", action="store_true", help="do nothing if this run already finished")
    z = sub.add_parser("zeroshot")
    z.add_argument("--model", required=True)
    z.add_argument("--batch-size", type=int, default=16)
    sub.add_parser("cadd")
    sub.add_parser("export-vcf")
    sub.add_parser("report")

    args = p.parse_args(argv)
    setup_logging(args.log_level)
    cfg = load_config(args.config)
    {"build-dataset": cmd_build_dataset, "train": cmd_train, "zeroshot": cmd_zeroshot, "cadd": cmd_cadd,
     "export-vcf": cmd_export_vcf, "report": cmd_report}[args.command](cfg, args)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/glm-variant-pathogenicity/configs/colab.yaml
# Reduced run for a free Google Colab T4 GPU (~2-3 hours in total).
# Differences from the full plan (configs/default.yaml), to report honestly:
#   * a random subset of each gene-level split (split itself unchanged)
#   * one seed, a fixed learning rate (no grid search), up to 3 epochs
#   * results and processed data are written to Google Drive so a
#     disconnected session can resume where it stopped
inherit: default.yaml

paths:
  raw: /content/data/raw
  processed: /content/drive/MyDrive/glm-variant-pathogenicity/processed
  runs: /content/drive/MyDrive/glm-variant-pathogenicity/runs
  reports: /content/drive/MyDrive/glm-variant-pathogenicity/report

data:
  clinvar_vcf: /content/data/raw/clinvar.vcf.gz
  reference_fasta: /content/data/raw/hg38.fa
  max_variants_per_split: {train: 24000, val: 3000, test: 4000}

training:
  batch_size: 32
  micro_batch_size: 16
  epochs: 3
  patience: 1
  learning_rate: 1.0e-4
  head_learning_rate: 1.0e-3
  num_workers: 2

cadd:
  whole_genome_tsv: https://kircherlab.bihealth.org/download/CADD/v1.7/GRCh38/whole_genome_SNVs.tsv.gz

evaluation:
  bootstrap: 1000
  seeds: [0]

In [ ]:
%cd /content/glm-variant-pathogenicity
!pip install -q -e . --no-deps
!glmvar --help | head -5

## 5. Download ClinVar and the GRCh38 genome
Skipped automatically if the dataset was already built in an earlier session.

In [ ]:
import os, gzip, pysam
DRIVE = "/content/drive/MyDrive/glm-variant-pathogenicity"
built = os.path.exists(f"{DRIVE}/processed/variants_w1024.parquet")
os.makedirs("/content/data/raw", exist_ok=True)
%cd /content/data/raw
if not built:
    !wget -q -nc https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz
    with gzip.open("clinvar.vcf.gz", "rt") as fh:
        for line in fh:
            if line.startswith("##fileDate"):
                print("ClinVar release:", line.strip()); open(f"{DRIVE}/clinvar_release.txt", "w").write(line); break
    if not os.path.exists("hg38.fa"):
        !wget -q -nc https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz
        !gunzip -f hg38.fa.gz
    pysam.faidx("hg38.fa")
    print("downloads done")
else:
    print("dataset already built - skipping downloads")
%cd /content/glm-variant-pathogenicity

## 6. Build the dataset (ClinVar labels, 1,024-bp windows, split by gene)

In [ ]:
if not built:
    !glmvar --config configs/colab.yaml build-dataset
import pandas as pd
df = pd.read_parquet(f"{DRIVE}/processed/variants_w1024.parquet")
print(df.groupby("split").label.agg(variants="size", pathogenic="sum"))
print("genes per split:", df.groupby("split").gene.nunique().to_dict())

## 7. Train the models
Fine-tuned (LoRA for DNABERT-2 and the Nucleotide Transformer, full fine-tuning for HyenaDNA), then the frozen-backbone versions. Each run takes roughly 5–30 minutes. A run that fails is reported and the others still run.

In [ ]:
runs = [("nt_v2_250m", "finetune"), ("dnabert2", "finetune"), ("hyenadna_medium", "finetune"),
        ("nt_v2_250m", "frozen"), ("dnabert2", "frozen"), ("hyenadna_medium", "frozen")]
failed = []
for model, mode in runs:
    print(f"\n==================== {model} / {mode} ====================", flush=True)
    !glmvar --config configs/colab.yaml train --model {model} --mode {mode} --seed 0 --skip-existing
    if _exit_code != 0:
        failed.append((model, mode))
print("\nfailed runs:", failed or "none")

## 8. Zero-shot scores (no training on ClinVar labels)

In [ ]:
import os
for model in ["nt_v2_250m", "dnabert2"]:
    if os.path.exists(f"{DRIVE}/runs/{model}_zeroshot/seed0/predictions_test.csv"):
        print(model, "already done"); continue
    !glmvar --config configs/colab.yaml zeroshot --model {model} --batch-size 32

## 9. CADD v1.7 scores for the validation and test variants
Read directly from the CADD server; only the needed pieces of the file are downloaded (about 5–20 minutes).

In [ ]:
if not os.path.exists(f"{DRIVE}/runs/cadd/seed0/predictions_test.csv"):
    %cd /content/data/raw
    !glmvar --config /content/glm-variant-pathogenicity/configs/colab.yaml cadd
    %cd /content/glm-variant-pathogenicity
else:
    print("CADD already done")

## 10. Results

In [ ]:
!glmvar --config configs/colab.yaml report
from IPython.display import Markdown, Image, display
display(Markdown(open(f"{DRIVE}/report/RESULTS.md").read()))
if os.path.exists(f"{DRIVE}/report/roc_test.png"):
    display(Image(f"{DRIVE}/report/roc_test.png", width=500))

In [ ]:
import json, glob
rows = []
for f in sorted(glob.glob(f"{DRIVE}/runs/*/seed0/summary.json")):
    s = json.load(open(f))
    rows.append({"run": f.split("/")[-3], "best_val_auroc": s.get("best_val_auroc"), "epochs": s.get("epochs_run"),
                 "gpu_hours": s.get("gpu_hours"), "peak_gpu_mem_gb": s.get("peak_gpu_mem_gb"),
                 "variants_per_s": s.get("variants_per_second"), "trainable_M": (s.get("trainable_params") or 0) / 1e6})
cost = pd.DataFrame(rows).round(4)
cost.to_csv(f"{DRIVE}/report/cost.csv", index=False)
cost

## 11. Download the results
This creates `results_bundle.zip` (tables, figures, run summaries; also saved in your Drive). Send it to Claude.

In [ ]:
!cd {DRIVE} && rm -f results_bundle.zip && zip -qr results_bundle.zip report clinvar_release.txt processed/dataset_stats.json runs -x "*.parquet"
from google.colab import files
files.download(f"{DRIVE}/results_bundle.zip")